# 🧠💻 Computer Vision Workshop — Session 1 Lab
### Classification · Detection · Segmentation — hands-on, in the cloud

Welcome! In this notebook you will **run real deep learning models**. No GPU on your laptop is required: everything runs on **Google Colab's free GPU** (Kaggle works too, and a lightweight CPU path is built in as a last resort).

**How to run this lab**
1. Colab: `Runtime → Change runtime type → T4 GPU`. Kaggle: `Settings → Accelerator → GPU`.
2. Run the cells **in order** (`Runtime → Run all` works too).
3. Watch for the callouts — those are the learning moments:
   - 🔎 **OBSERVE** — what to look at in the output
   - ✏️ **TWEAK** — change one thing and re-run
   - 💬 **DISCUSS** — questions for you / your group

| Part | You will... | Models used |
|---|---|---|
| 0 | Set up the environment & check the GPU | — |
| 1 | **Classification** — train from scratch, then fine-tune | SimpleCNN, ResNet18 |
| 2 | **Detection** — find objects and draw boxes | Faster R-CNN (pretrained on COCO) |
| 3 | **Segmentation** — color every pixel by class | DeepLabV3 (pretrained on COCO/VOC) |
| 🎉 | **Fun Zone** — one image through all three tasks | all of the above |
| ⭐ | *Optional bonus* — train a tiny U-Net | TinyUNet |

> 💡 **No GPU right now?** In the next cell set `MODE = "fast_cpu"`. Same lab, smaller data, slower — works on any laptop.
>
> ⚠️ **Colab says "You are not connected to a GPU"?** Check `Runtime → Change runtime type → T4 GPU`, re-run from the top. If Colab has no GPUs available, switch to **Kaggle** (`Settings → Accelerator → GPU`, turn **Internet ON**) or use `fast_cpu`.


## ⚙️ Part 0 — Setup

We first choose the runtime mode, check whether a GPU is present, and download a small **image gallery** we will use throughout the lab.

Everything after this point is designed to work in all three environments:
**Colab (GPU) → Kaggle (GPU) → your own laptop (CPU)**.

In [ ]:
# ============================================================
# ⚙️ CONFIG — choose your runtime mode, then keep going
# ============================================================
MODE = "gpu"  #@param ["gpu", "fast_cpu"]

import os, sys, time, json, shutil, random
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

GALLERY_DIR = Path("gallery"); OUTPUT_DIR = Path("outputs")
GALLERY_DIR.mkdir(exist_ok=True); OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Mode selected: {MODE}")

In [ ]:
# ---- Environment check -------------------------------------------------
import torch
GPU_AVAILABLE = torch.cuda.is_available()

if MODE == "gpu" and not GPU_AVAILABLE:
    print("⚠️  You selected 'gpu' but no GPU was found.")
    print("    • Colab : Runtime → Change runtime type → T4 GPU, then re-run this cell.")
    print("    • Kaggle: Settings → Accelerator → GPU.")
    print("    → Falling back to 'fast_cpu' so the lab still works.\n")
    MODE = "fast_cpu"

DEVICE = "cuda" if (MODE == "gpu" and GPU_AVAILABLE) else "cpu"

import platform
print(f"Python    : {platform.python_version()}")
print(f"PyTorch   : {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU       : {torch.cuda.get_device_name(0)} ✅")
else:
    print(f"Device    : CPU (lightweight path — smaller data, fewer epochs)")
print(f"CPU cores : {os.cpu_count()}")

In [ ]:
# ---- Mode-dependent settings -------------------------------------------
# fast_cpu = smaller subsets, fewer epochs, smaller images, tiny U-Net.
CFG = {
    "gpu":      dict(train_per_class=2000, test_per_class=300, epochs_scratch=5,
                     epochs_ft=3, batch=64, ft_size=128, unet_epochs=5, unet_images=300),
    "fast_cpu": dict(train_per_class=500,  test_per_class=200, epochs_scratch=2,
                     epochs_ft=1, batch=32, ft_size=64,  unet_epochs=1, unet_images=120),
}[MODE]

for k, v in CFG.items():
    print(f"{k:>16} : {v}")

In [ ]:
# ---- Install anything that is missing (no-op on Colab) ------------------
import importlib.util, subprocess

def ensure(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    else:
        print(f"✓ {pkg} already available")

ensure("scikit-learn")
ensure("Pillow", "PIL")

In [ ]:
%matplotlib inline
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report

try:
    from tqdm.auto import tqdm
except ImportError:                      # very small fallback
    def tqdm(x, **kw): return x

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(SEED)
print("torchvision:", torchvision.__version__)

### 🖼️ Our image gallery

The next cell downloads a handful of **freely usable test images**. They are chosen so that all three tasks have something interesting to find:

| Image | Good for |
|---|---|
| `bus.jpg` — bus + people on a street | detection & segmentation of `bus`, `person` |
| `cats.jpg` — two cats on a couch | detection (`cat`, `remote`, `couch`) |
| `dog.jpg`, `bear.jpg`, `chelsea.png` — animals | classification & segmentation |
| `baseball.jpg`, `skier.jpg` — sports scenes | detection of `person` + equipment |
| `living_room.jpg`, `bedroom.jpg` — indoor scenes | segmentation of furniture |
| `astronaut.png` — a person | everything! |

If a download fails (rare), the notebook simply skips it — you can also upload your own image later in the Fun Zone.

In [ ]:
GALLERY = [
    ("bus.jpg",         "https://ultralytics.com/images/bus.jpg"),
    ("zidane.jpg",      "https://ultralytics.com/images/zidane.jpg"),
    ("dog.jpg",         "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"),
    ("cats.jpg",        "http://images.cocodataset.org/val2017/000000039769.jpg"),
    ("bear.jpg",        "http://images.cocodataset.org/val2017/000000000285.jpg"),
    ("baseball.jpg",    "http://images.cocodataset.org/val2017/000000002153.jpg"),
    ("skier.jpg",       "http://images.cocodataset.org/val2017/000000000785.jpg"),
    ("living_room.jpg", "http://images.cocodataset.org/val2017/000000000139.jpg"),
    ("bedroom.jpg",     "http://images.cocodataset.org/val2017/000000000632.jpg"),
    ("astronaut.png",   "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.22.0/skimage/data/astronaut.png"),
    ("chelsea.png",     "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.22.0/skimage/data/chelsea.png"),
]

import urllib.request

def download(url, dest, timeout=60):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest, "wb") as f:
            shutil.copyfileobj(r, f)
        return True
    except Exception as e:
        print(f"   ✗ {dest.name}: {e}")
        return False

missing = []
for name, url in GALLERY:
    dest = GALLERY_DIR / name
    if dest.exists() and dest.stat().st_size > 1000:
        continue
    print(f"⬇️  {name}")
    if not download(url, dest):
        missing.append(name)

n_ok = len([p for p in GALLERY_DIR.iterdir() if p.is_file()])
print(f"\nGallery ready: {n_ok} images in '{GALLERY_DIR}/'")
if missing:
    print("Missing (will be skipped):", missing)

In [ ]:
def show_gallery():
    paths = sorted(p for p in GALLERY_DIR.iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    cols = 4
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 2.6 * rows))
    for ax, p in zip(axes.ravel(), paths):
        ax.imshow(Image.open(p))
        ax.set_title(p.name, fontsize=9)
        ax.axis("off")
    for ax in axes.ravel()[len(paths):]:
        ax.axis("off")
    plt.tight_layout()
    return fig

fig = show_gallery()
fig.savefig(OUTPUT_DIR / "gallery_preview.png", bbox_inches="tight")
plt.show()

## 🧩 Part 1 — Image Classification

**Classification** answers: *"what is in this image?"* — one label per image.

We will do it **twice**, and that contrast is the most important lesson of the day:

1. **From scratch** — we build a small CNN and train it on our own data. It starts knowing *nothing*.
2. **Transfer learning** — we take a ResNet18 that already learned from 1.2 million ImageNet photos, freeze its "eyes", and train only a fresh decision layer for our 3 classes.

Then we compare. This is exactly how real projects are built.

### 1a. Dataset — CIFAR-10 (3 classes)

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) is a classic benchmark: 60,000 images, 32×32 pixels, 10 classes. To keep training fast we keep only **3 visually distinct classes** — ✈️ `airplane`, 🐱 `cat`, 🐶 `dog` — and a subset of each.

🔎 **OBSERVE**: 32×32 is *tiny*. Each image is a `3 × 32 × 32` tensor = 3,072 numbers. That is all a computer "sees".

In [ ]:
CLASSES = {0: "airplane", 3: "cat", 5: "dog"}   # CIFAR-10 class ids we keep
NUM_CLASSES = len(CLASSES)
class_names = [CLASSES[c] for c in sorted(CLASSES)]
print("Our classes:", class_names)

def subset_indices(dataset, classes, per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    by_class = {c: [] for c in classes}
    for i, t in enumerate(dataset.targets):
        if t in by_class and len(by_class[t]) < per_class:
            by_class[t].append(i)
        if all(len(v) >= per_class for v in by_class.values()):
            break
    idx = [i for c in classes for i in by_class[c]]
    rng.shuffle(idx)
    return idx

class RemapLabels(Dataset):
    # Keep only our classes and remap their labels to 0..K-1.
    def __init__(self, base, mapping):
        self.base, self.mapping = base, mapping
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return x, self.mapping[y]

class ApplyTransform(Dataset):
    # Apply a torchvision transform to a PIL dataset on-the-fly.
    def __init__(self, base, tf):
        self.base, self.tf = base, tf
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        if self.tf is not None:
            x = self.tf(x)
        return x, y

CIFAR_MEAN, CIFAR_STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

train_full = CIFAR10(root="data", train=True,  download=True)
test_full  = CIFAR10(root="data", train=False, download=True)

train_idx = subset_indices(train_full, CLASSES.keys(), CFG["train_per_class"])
test_idx  = subset_indices(test_full,  CLASSES.keys(), CFG["test_per_class"])
mapping = {orig: new for new, orig in enumerate(CLASSES)}

tf_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_ds = ApplyTransform(RemapLabels(Subset(train_full, train_idx), mapping), tf_cifar)
test_ds  = ApplyTransform(RemapLabels(Subset(test_full,  test_idx),  mapping), tf_cifar)

nw = 2 if DEVICE == "cuda" else 0
train_loader = DataLoader(train_ds, batch_size=CFG["batch"], shuffle=True,
                          num_workers=nw, pin_memory=(DEVICE == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch"], shuffle=False,
                          num_workers=nw, pin_memory=(DEVICE == "cuda"))

print(f"Train images: {len(train_ds)} | Test images: {len(test_ds)}")
for c, name in zip(sorted(CLASSES), class_names):
    n_tr = sum(1 for i in train_idx if train_full.targets[i] == c)
    print(f"  {name:<9}: {n_tr} training images")

In [ ]:
def show_batch(loader, n=8, title="Training samples"):
    xb, yb = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(1.6 * n, 2.3))
    for ax, im, lab in zip(axes, xb[:n], yb[:n]):
        ax.imshow((im.permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
        ax.set_title(class_names[lab], fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    return fig

fig = show_batch(train_loader, title="CIFAR-10 training samples (3 classes)")
fig.savefig(OUTPUT_DIR / "cifar_samples.png", bbox_inches="tight")
plt.show()

### 1b. A CNN from scratch

Our little network has **3 convolution layers** (exactly the structure from the lecture) and a small classifier head:

| Block | Operation | Output |
|---|---|---|
| `conv1` | 3→16 channels, 3×3, ReLU, MaxPool | 16×16 |
| `conv2` | 16→32 channels, 3×3, ReLU, MaxPool | 8×8 |
| `conv3` | 32→64 channels, 3×3, ReLU, MaxPool | 4×4 |
| `fc1` | 64·4·4 = 1024 → 128, ReLU | 128 |
| `fc2` | 128 → 3 (logits) | 3 |

🔎 **OBSERVE**: the spatial size *shrinks* (32→16→8→4) while the number of channels *grows* (3→16→32→64). The network is trading space for meaning.

In [ ]:
class SimpleCNN(nn.Module):
    # A small 3-convolution CNN we will train from scratch.
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),   # 32 -> 16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16 -> 8
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8  -> 4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

t, tr = count_params(SimpleCNN())
print(f"SimpleCNN parameters: {t:,} total / {tr:,} trainable")

In [ ]:
# ---- Reusable training / evaluation helpers -----------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        out = model(x.to(DEVICE))
        ps.append(out.argmax(1).cpu()); ys.append(y)
    ys = torch.cat(ys).numpy(); ps = torch.cat(ps).numpy()
    return {"accuracy": float((ys == ps).mean()), "y_true": ys, "y_pred": ps}

def train_model(model, train_loader, test_loader, epochs=5, lr=1e-3, tag="model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    history = {"train_loss": [], "train_acc": [], "test_acc": []}
    model.to(DEVICE)
    for epoch in range(epochs):
        model.train()
        running_loss = correct = total = 0
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch+1}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
            if hasattr(pbar, "set_postfix"):
                pbar.set_postfix(loss=f"{loss.item():.3f}")
        tr_loss, tr_acc = running_loss / total, correct / total
        te_acc = evaluate(model, test_loader)["accuracy"]
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["test_acc"].append(te_acc)
        print(f"[{tag}] epoch {epoch+1}/{epochs} — train loss {tr_loss:.3f} | "
              f"train acc {tr_acc*100:4.1f}% | test acc {te_acc*100:4.1f}%")
    return history

def plot_history(histories, title="Training"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in histories.items():
        e = range(1, len(h["train_loss"]) + 1)
        axes[0].plot(e, h["train_loss"], marker="o", label=name)
        axes[1].plot(e, [a * 100 for a in h["test_acc"]], marker="o", label=name)
    axes[0].set(xlabel="epoch", ylabel="loss", title="Training loss"); axes[0].grid(alpha=.3); axes[0].legend()
    axes[1].set(xlabel="epoch", ylabel="accuracy (%)", title="Test accuracy"); axes[1].grid(alpha=.3); axes[1].legend()
    fig.suptitle(title)
    plt.tight_layout()
    return fig

In [ ]:
scratch_model = SimpleCNN().to(DEVICE)
t0 = time.time()
hist_scratch = train_model(scratch_model, train_loader, test_loader,
                           epochs=CFG["epochs_scratch"], lr=1e-3, tag="scratch")
print(f"\n⏱️ Trained from scratch in {time.time()-t0:.0f}s")

fig = plot_history({"CNN from scratch": hist_scratch}, "Training a CNN from scratch")
fig.savefig(OUTPUT_DIR / "curves_scratch.png", bbox_inches="tight")
plt.show()

✏️ **TWEAK (optional, 2 min)** — pick one and re-run:
- Increase `epochs_scratch` to 10. Does test accuracy keep improving?
- Change `lr=1e-3` to `1e-2` (too big?) or `1e-4` (too slow?). Watch the loss curve.
- Add `nn.Dropout(0.3)` before the last `Linear` in `SimpleCNN`.

💬 **DISCUSS**: the loss goes down and accuracy goes up — but what is the network *actually learning*? (Hint: look at the filters/feature maps from the lecture.)

### 1c. Transfer learning — stand on the shoulders of giants

ResNet18 was trained on **ImageNet** (1.2M photos, 1000 classes). Its convolutional layers already know edges, textures, shapes, fur, wheels...

We will:
1. **Freeze** every pretrained layer (`requires_grad = False`).
2. **Replace** the final layer with a fresh `Linear(512 → 3)`.
3. Train **only that head** for a few epochs — a few seconds of work.

🔎 **OBSERVE**: because we resize CIFAR images up to `CFG["ft_size"]`, ResNet sees its familiar kind of input.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

ft_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# 1) Freeze the pretrained "eyes"
for p in ft_model.parameters():
    p.requires_grad = False

# 2) Replace the classifier head with a fresh one for our 3 classes
ft_model.fc = nn.Linear(ft_model.fc.in_features, NUM_CLASSES)

t, tr = count_params(ft_model)
print(f"ResNet18: {t:,} parameters total / {tr:,} trainable  ← only the new head trains!")

In [ ]:
# ResNet likes a bigger input than 32x32 -> resize for fine-tuning
ft_size = CFG["ft_size"]
tf_ft = transforms.Compose([
    transforms.Resize((ft_size, ft_size)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

ft_train_ds = ApplyTransform(RemapLabels(Subset(train_full, train_idx), mapping), tf_ft)
ft_test_ds  = ApplyTransform(RemapLabels(Subset(test_full,  test_idx),  mapping), tf_ft)
ft_train_loader = DataLoader(ft_train_ds, batch_size=CFG["batch"], shuffle=True,
                             num_workers=nw, pin_memory=(DEVICE == "cuda"))
ft_test_loader  = DataLoader(ft_test_ds,  batch_size=CFG["batch"], shuffle=False,
                             num_workers=nw, pin_memory=(DEVICE == "cuda"))

print(f"Fine-tuning at {ft_size}x{ft_size} on {len(ft_train_ds)} images")
t0 = time.time()
hist_ft = train_model(ft_model, ft_train_loader, ft_test_loader,
                      epochs=CFG["epochs_ft"], lr=1e-3, tag="resnet18-ft")
print(f"\n⏱️ Fine-tuned in {time.time()-t0:.0f}s")

fig = plot_history({"CNN from scratch": hist_scratch, "ResNet18 fine-tuned": hist_ft},
                   "Custom CNN vs Transfer Learning")
fig.savefig(OUTPUT_DIR / "curves_compare.png", bbox_inches="tight")
plt.show()

💬 **DISCUSS**
- Compare the final test accuracies. Which won? By how much?
- The scratch model had **~110k trainable parameters**; the fine-tuned head has only **~1.5k**. Why does the smaller one win?
- When would training from scratch still be the right choice? (Hint: think of domains far from everyday photos.)

### 1d. Evaluation — reading a confusion matrix

Accuracy alone hides *how* a model is wrong. A **confusion matrix** shows which classes get mixed up — in our case the classic **cat ↔ dog** problem.

We evaluate the fine-tuned model (and compare with the scratch one).

In [ ]:
res_scratch = evaluate(scratch_model, test_loader)
res_ft      = evaluate(ft_model, ft_test_loader)
print(f"Scratch CNN test accuracy : {res_scratch['accuracy']*100:.1f}%")
print(f"Fine-tuned  test accuracy : {res_ft['accuracy']*100:.1f}%")

cm = confusion_matrix(res_ft["y_true"], res_ft["y_pred"])

def plot_confusion(cm, labels, title):
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    im = ax.imshow(cm, cmap="Blues")
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
           xticklabels=labels, yticklabels=labels,
           xlabel="Predicted", ylabel="True", title=title)
    plt.colorbar(im, fraction=0.046)
    plt.tight_layout()
    return fig

fig = plot_confusion(cm, class_names, "Fine-tuned ResNet18 — confusion matrix")
fig.savefig(OUTPUT_DIR / "confusion_matrix_ft.png", bbox_inches="tight")
plt.show()

print(classification_report(res_ft["y_true"], res_ft["y_pred"], target_names=class_names))

🔎 **OBSERVE** the report: `precision`, `recall` and `f1-score` per class. A row of the matrix that is "smeared" into another class tells you exactly which categories the model confuses — and hints at what data or techniques to add next (more data? augmentation? a different loss?).

💬 **DISCUSS**: `cat` vs `dog` — which pairs of classes should be *easiest* to separate? Check whether the matrix agrees.

In [ ]:
# ---- Quick test: our fine-tuned model on gallery images -----------------
def classify_image(path, model, class_names, size=32, topk=3):
    tf = transforms.Compose([transforms.Resize((size, size)),
                             transforms.ToTensor(),
                             transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
    img = Image.open(path).convert("RGB")
    x = tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = model(x).softmax(1)[0]
    vals, idx = probs.topk(topk)
    return img, [(class_names[i], float(v)) for v, i in zip(vals, idx)]

for name in ["chelsea.png", "dog.jpg", "bus.jpg"]:
    p = GALLERY_DIR / name
    if not p.exists():
        continue
    _, top = classify_image(p, ft_model, class_names, size=ft_size)
    pretty = " | ".join(f"{lab} {prob*100:.1f}%" for lab, prob in top)
    print(f"{name:<14} → {pretty}")

print("\n⚠️ Remember: this model only knows 3 classes. In the Fun Zone we use a")
print("   1000-class ImageNet model — much more fun for arbitrary photos!")

## 📦 Part 2 — Object Detection

**Detection** answers: *"what is where?"* — a list of **boxes**, each with a class and a confidence.

Two ideas from the lecture come alive here:

- **IoU (Intersection over Union)** — how much two boxes overlap: `IoU = area(A ∩ B) / area(A ∪ B)`. Used both to score predictions and to clean them up.
- **NMS (Non-Maximum Suppression)** — a detector proposes *many* overlapping boxes for the same object. NMS keeps the most confident one and deletes the rest.

We use **Faster R-CNN** with a MobileNet backbone, pretrained on **COCO** (80 everyday classes). We will *not* train it — training detectors from scratch takes days and huge labeled datasets; in practice you **download pretrained weights** and run inference. That is the skill we practice now.

In [ ]:
from torchvision.models.detection import (fasterrcnn_mobilenet_v3_large_320_fpn,
                                          FasterRCNN_MobileNet_V3_Large_320_FPN_Weights)

DET_WEIGHTS = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.COCO_V1
DET_TRANSFORM = DET_WEIGHTS.transforms()
COCO_NAMES = DET_WEIGHTS.meta["categories"]
DET_THRESHOLD = 0.55

detector = fasterrcnn_mobilenet_v3_large_320_fpn(weights=DET_WEIGHTS).eval().to(DEVICE)

@torch.no_grad()
def detect_image(path, score_threshold=DET_THRESHOLD, max_boxes=20):
    img = Image.open(path).convert("RGB")
    x = DET_TRANSFORM(img).to(DEVICE)
    out = detector([x])[0]
    keep = out["scores"] >= score_threshold
    boxes, labels, scores = out["boxes"][keep][:max_boxes], out["labels"][keep][:max_boxes], out["scores"][keep][:max_boxes]
    dets = [{"box": [round(float(v), 1) for v in b],
             "label": COCO_NAMES[l],
             "score": round(float(s), 3)}
            for b, l, s in zip(boxes, labels, scores)]
    return img, dets

def draw_detections(img, dets):
    im = img.copy()
    d = ImageDraw.Draw(im)
    for det in dets:
        x1, y1, x2, y2 = det["box"]
        d.rectangle([x1, y1, x2, y2], outline=(255, 40, 40), width=3)
        d.text((x1 + 4, max(0, y1 - 14)), f'{det["label"]} {det["score"]:.2f}', fill=(255, 255, 0))
    return im

print(f"Detector ready ({len(COCO_NAMES)} classes). Try threshold {DET_THRESHOLD}.")

In [ ]:
def show_detections(names, cols=2):
    paths = [GALLERY_DIR / n for n in names if (GALLERY_DIR / n).exists()]
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6.2 * cols, 5.0 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, p in zip(axes, paths):
        img, dets = detect_image(p)
        ax.imshow(draw_detections(img, dets))
        ax.set_title(f"{p.name} — {len(dets)} object(s)", fontsize=10)
        ax.axis("off")
        print(f"{p.name}: " + (", ".join(f"{d['label']} ({d['score']:.2f})" for d in dets) or "nothing above threshold"))
    for ax in axes[len(paths):]:
        ax.axis("off")
    plt.tight_layout()
    return fig

fig = show_detections(["bus.jpg", "baseball.jpg", "zidane.jpg", "dog.jpg"])
fig.savefig(OUTPUT_DIR / "detection_gallery.png", bbox_inches="tight")
plt.show()

✏️ **TWEAK**: change `DET_THRESHOLD` (e.g. `0.30` → many boxes, some nonsense; `0.90` → only the sure ones). Re-run the cell. This single number controls *precision vs recall*.

### NMS, by hand

Below we simulate what the detector produces *before* cleanup: several overlapping boxes for the same object, each with a score. `torchvision.ops.nms` keeps the best box and removes any box overlapping it more than `iou_threshold`.

In [ ]:
import torchvision

boxes = torch.tensor([[30, 30, 180, 180], [40, 40, 175, 175], [35, 50, 170, 190],
                      [200, 60, 320, 200], [205, 65, 315, 195]], dtype=torch.float32)
scores = torch.tensor([0.95, 0.90, 0.80, 0.88, 0.70])
keep = torchvision.ops.nms(boxes, scores, iou_threshold=0.5)
print("Kept boxes:", keep.tolist(), "(indices) — everything else was suppressed")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, idxs, title in [(axes[0], range(len(boxes)), "Before NMS (raw proposals)"),
                        (axes[1], keep.tolist(), "After NMS (one box per object)")]:
    ax.set(xlim=(0, 360), ylim=(240, 0), title=title)
    for i in idxs:
        x1, y1, x2, y2 = boxes[i]
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                   edgecolor="red", lw=2, alpha=0.9))
        ax.text(x1 + 3, y1 + 12, f"{scores[i]:.2f}", color="red", fontsize=9)
fig.suptitle("Non-Maximum Suppression (IoU threshold = 0.5)")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "nms_demo.png", bbox_inches="tight")
plt.show()

💬 **DISCUSS**: why does the *left* picture exist at all — why does a detector propose multiple boxes for the same object? (Think about the anchor boxes / region proposals from the lecture.)

## 🎨 Part 3 — Image Segmentation

**Segmentation** answers: *"which pixels belong to what?"* — a class label for **every pixel**.

We use **DeepLabV3** (MobileNet backbone), pretrained on COCO/VOC — it knows 21 classes (person, dog, cat, bus, sofa...). Notice the architecture from the lecture:

- an **encoder** that shrinks the image and extracts meaning,
- a **decoder** that brings it back to full resolution,
- **skip connections** that carry fine detail across.

And we will *not* train it — same reason as detection. But there is an optional bonus at the end where you **do** train a tiny U-Net, so you see what that looks like.

🔎 **OBSERVE**: the mask is one integer per pixel — class 0 is background, 15 is person, etc.

In [ ]:
from torchvision.models.segmentation import (deeplabv3_mobilenet_v3_large,
                                              DeepLabV3_MobileNet_V3_Large_Weights)

SEG_WEIGHTS = DeepLabV3_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1
SEG_TRANSFORM = SEG_WEIGHTS.transforms()
VOC_NAMES = SEG_WEIGHTS.meta["categories"]

# Standard VOC color palette (RGB) for the 21 classes
VOC_COLORS = [(0,0,0),(128,0,0),(0,128,0),(128,128,0),(0,0,128),(128,0,128),(0,128,128),
              (128,128,128),(64,0,0),(192,0,0),(64,128,0),(192,128,0),(64,0,128),(192,0,128),
              (64,128,128),(192,128,128),(0,64,0),(128,64,0),(0,192,0),(128,192,0),(0,64,128)]

segmenter = deeplabv3_mobilenet_v3_large(weights=SEG_WEIGHTS).eval().to(DEVICE)

@torch.no_grad()
def segment_image(path):
    img = Image.open(path).convert("RGB")
    x = SEG_TRANSFORM(img).to(DEVICE)
    logits = segmenter(x.unsqueeze(0))["out"][0]                 # (21, h, w)
    mask = logits.argmax(0).cpu().numpy().astype(np.uint8)       # (h, w)
    mask = np.array(Image.fromarray(mask).resize(img.size, Image.NEAREST))
    return img, mask

def overlay_mask(img, mask, alpha=0.55):
    colors = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for c in np.unique(mask):
        colors[mask == c] = VOC_COLORS[c]
    base = np.array(img).astype(np.float32)
    m = (mask > 0)[..., None]
    blended = np.where(m, (1 - alpha) * base + alpha * colors, base).astype(np.uint8)
    return Image.fromarray(blended)

def mask_coverage(mask):
    total = mask.size
    counts = {}
    for c in np.unique(mask):
        if c == 0:
            continue
        counts[VOC_NAMES[c]] = 100.0 * np.sum(mask == c) / total
    return counts

print(f"Segmenter ready — {len(VOC_NAMES)} classes (including background).")

In [ ]:
def show_segmentations(names, cols=2):
    paths = [GALLERY_DIR / n for n in names if (GALLERY_DIR / n).exists()]
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols * 2, figsize=(5.2 * cols * 2, 4.6 * rows))
    axes = np.atleast_1d(axes).reshape(rows, cols * 2)
    for k, p in enumerate(paths):
        r, c = divmod(k, cols)
        img, mask = segment_image(p)
        axes[r, c * 2].imshow(img); axes[r, c * 2].set_title(f"{p.name}", fontsize=9)
        axes[r, c * 2 + 1].imshow(overlay_mask(img, mask)); axes[r, c * 2 + 1].set_title("predicted mask", fontsize=9)
        axes[r, c * 2].axis("off"); axes[r, c * 2 + 1].axis("off")
        cov = mask_coverage(mask)
        top = sorted(cov.items(), key=lambda kv: -kv[1])[:3]
        print(f"{p.name:<16}: " + ", ".join(f"{k2} {v:.1f}%" for k2, v in top))
    for ax in axes.ravel()[len(paths) * 2:]:
        ax.axis("off")
    plt.tight_layout()
    return fig

fig = show_segmentations(["bus.jpg", "cats.jpg", "chelsea.png", "living_room.jpg"])
fig.savefig(OUTPUT_DIR / "segmentation_gallery.png", bbox_inches="tight")
plt.show()

🔎 **OBSERVE**: compare a box from Part 2 with a mask here. The box says *"a dog is roughly here"*; the mask traces the actual fur boundary. That extra precision is why segmentation is used in medical imaging, robotics, and photo editing.

💬 **DISCUSS**: where would a rectangle be *useless* but a mask essential? (Think: tumors, road lanes, product cutouts.)

## 🎉 The Fun Zone — your image, all three tasks

Now it is your turn. Pick **any image** — one from our gallery or one of your own — and we run the full pipeline on it:

1. 🏷️ **Classification** (1000-class ImageNet model — it knows almost anything)
2. 📦 **Detection** (Faster R-CNN, 80 COCO classes)
3. 🎨 **Segmentation** (DeepLabV3, 21 VOC classes)

Everything gets saved into `outputs/` so you can take it home.

> **In Colab**: set `UPLOAD_MY_OWN = True` below and a *Choose files* button appears.
> **Elsewhere**: type a path in `MY_IMAGE`, or leave both empty to use `bus.jpg`.

In [ ]:
#@title 🎯 Pick your image { display-mode: "form" }
MY_IMAGE = ""  #@param {type:"string"}
UPLOAD_MY_OWN = False  #@param {type:"boolean"}

FUN_IMAGE = None
if UPLOAD_MY_OWN and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()                     # opens the file picker
    if uploaded:
        name = next(iter(uploaded))
        p = Path("uploads") / name
        p.parent.mkdir(exist_ok=True)
        p.write_bytes(uploaded[name])
        FUN_IMAGE = p
elif MY_IMAGE.strip():
    FUN_IMAGE = Path(MY_IMAGE.strip())
    if not FUN_IMAGE.exists():
        print(f"⚠️ {FUN_IMAGE} not found — using bus.jpg instead")
        FUN_IMAGE = None
if FUN_IMAGE is None:
    FUN_IMAGE = GALLERY_DIR / "bus.jpg"

assert FUN_IMAGE.exists(), f"Image not found: {FUN_IMAGE}"
print("🎯 Fun Zone image:", FUN_IMAGE)
plt.figure(figsize=(5, 5)); plt.imshow(Image.open(FUN_IMAGE)); plt.axis("off")
plt.title(FUN_IMAGE.name); plt.show()

In [ ]:
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

IMG_WEIGHTS = MobileNet_V3_Large_Weights.IMAGENET1K_V2
IMG_TF = IMG_WEIGHTS.transforms()
IMG_NAMES = IMG_WEIGHTS.meta["categories"]
imagenet_model = mobilenet_v3_large(weights=IMG_WEIGHTS).eval().to(DEVICE)

@torch.no_grad()
def classify_imagenet(path, topk=5):
    img = Image.open(path).convert("RGB")
    x = IMG_TF(img).unsqueeze(0).to(DEVICE)
    probs = imagenet_model(x).softmax(1)[0]
    vals, idx = probs.topk(topk)
    return img, [(IMG_NAMES[i], float(v)) for v, i in zip(vals, idx)]

print("ImageNet classifier ready (1000 classes).")

In [ ]:
import csv

def save_detections_csv(dets, path):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["label", "score", "x1", "y1", "x2", "y2"])
        for d_ in dets:
            w.writerow([d_["label"], d_["score"], *d_["box"]])

stem = FUN_IMAGE.stem

# ---- 1) Classification ---------------------------------------------------
img_c, top5 = classify_imagenet(FUN_IMAGE)
print("🏷️  CLASSIFICATION (ImageNet, top-5):")
for lab, p in top5:
    print(f"     {lab:<30} {p*100:5.1f}%")
with open(OUTPUT_DIR / f"fun_{stem}_classification.txt", "w") as f:
    f.write("\n".join(f"{lab}\t{p*100:.1f}%" for lab, p in top5))

# ---- 2) Detection ---------------------------------------------------------
img_d, dets = detect_image(FUN_IMAGE)
annotated = draw_detections(img_d, dets)
print(f"\n📦 DETECTION: {len(dets)} object(s) with score ≥ {DET_THRESHOLD}")
for d_ in dets[:10]:
    print(f"     {d_['label']:<15} {d_['score']:.2f}   box={d_['box']}")
annotated.save(OUTPUT_DIR / f"fun_{stem}_detection.png")
save_detections_csv(dets, OUTPUT_DIR / f"fun_{stem}_detections.csv")

# ---- 3) Segmentation -------------------------------------------------------
img_s, mask = segment_image(FUN_IMAGE)
overlay = overlay_mask(img_s, mask)
cov = mask_coverage(mask)
print(f"\n🎨 SEGMENTATION coverage:")
for k, v in sorted(cov.items(), key=lambda kv: -kv[1])[:6]:
    print(f"     {k:<15} {v:5.1f}%")
overlay.save(OUTPUT_DIR / f"fun_{stem}_segmentation.png")
np.save(OUTPUT_DIR / f"fun_{stem}_mask.npy", mask)

# ---- Combined poster --------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
axes[0, 0].imshow(img_c); axes[0, 0].set_title("Input"); axes[0, 0].axis("off")
axes[0, 1].barh([lab for lab, _ in top5][::-1], [p * 100 for _, p in top5][::-1], color="tab:blue")
axes[0, 1].set_title("Classification (top-5 %)"); axes[0, 1].set_xlim(0, 100)
axes[1, 0].imshow(annotated); axes[1, 0].set_title(f"Detection ({len(dets)} objects)"); axes[1, 0].axis("off")
axes[1, 1].imshow(overlay); axes[1, 1].set_title("Segmentation"); axes[1, 1].axis("off")
fig.suptitle(f"One image, three models — {FUN_IMAGE.name}", fontsize=13)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"fun_{stem}_ALL.png", bbox_inches="tight")
plt.show()

print(f"\n💾 Saved to {OUTPUT_DIR}/: fun_{stem}_classification.txt, fun_{stem}_detection.png,")
print(f"   fun_{stem}_detections.csv, fun_{stem}_segmentation.png, fun_{stem}_mask.npy, fun_{stem}_ALL.png")

💬 **DISCUSS — compare the three answers for the *same* image:**
- Did the classifier's top guess agree with the detector's labels? Did the mask cover what the box found?
- Find one case where the **detector was fooled** (a box that is wrong or missing). Why might that happen?
- The models are *different sizes and speeds* — check the timing. Which would you deploy on a phone? A server?

✏️ **TWEAK**: try a different gallery image (type its name in `MY_IMAGE`), or your own photo — screenshots, pets, your desk. Try `bus.jpg` vs `chelsea.png` and compare the mask coverage.

## ⭐ Optional bonus — train a tiny U-Net (homework)

You have now *used* a segmentation model. In this bonus you will **train** one — a tiny U-Net — on a synthetic dataset of colored shapes. The point is to see the pieces you already met in the lecture in code:

- **skip connections**: `torch.cat([up, encoder_feature])`
- **per-pixel loss**: `BCEWithLogitsLoss` on a mask

It trains in seconds on a GPU (a minute or two on CPU). Do it if you finish early or at home.

In [ ]:
def make_shapes(n, size=64, seed=SEED):
    # Synthetic dataset: colored circles/rectangles on noisy backgrounds + binary masks.
    rng = np.random.default_rng(seed)
    imgs, masks = [], []
    for _ in range(n):
        img = (rng.normal(0.6, 0.10, (size, size, 3)) * 255).clip(0, 255).astype(np.uint8)
        canvas = Image.fromarray(img)
        draw = ImageDraw.Draw(canvas)
        mask = np.zeros((size, size), np.float32)
        for _ in range(rng.integers(1, 4)):
            r = int(rng.integers(6, 15))
            cx, cy = rng.integers(r, size - r, 2)
            color = tuple(int(c) for c in rng.integers(0, 255, 3))
            m = Image.new("L", (size, size), 0)
            md_ = ImageDraw.Draw(m)
            box = [cx - r, cy - r, cx + r, cy + r]
            if rng.random() < 0.5:
                draw.ellipse(box, fill=color); md_.ellipse(box, fill=1)
            else:
                draw.rectangle(box, fill=color); md_.rectangle(box, fill=1)
            mask = np.maximum(mask, np.array(m, dtype=np.float32))
        imgs.append(np.array(canvas, np.float32) / 255.0)
        masks.append(mask)
    X = torch.tensor(np.stack(imgs)).permute(0, 3, 1, 2)
    M = torch.tensor(np.stack(masks)).unsqueeze(1)
    return X, M

X, M = make_shapes(6)
fig, axes = plt.subplots(2, 6, figsize=(14, 4.4))
for i in range(6):
    axes[0, i].imshow(X[i].permute(1, 2, 0)); axes[0, i].axis("off")
    axes[1, i].imshow(M[i, 0], cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("image", loc="left"); axes[1, 0].set_title("mask", loc="left")
fig.suptitle("Synthetic shapes — our tiny segmentation dataset")
plt.tight_layout(); plt.show()

In [ ]:
class TinyUNet(nn.Module):
    # A minimal U-Net: 2 down blocks, bottleneck, 2 up blocks with skip connections.
    def __init__(self, base=16):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(3, base, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(base, base, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(base, base * 2, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(base * 2, base * 2, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(nn.Conv2d(base * 2, base * 4, 3, padding=1), nn.ReLU())
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(base * 4, base * 2, 3, padding=1), nn.ReLU())
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(base * 2, base, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))   # ← skip connection
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # ← skip connection
        return self.out(d1)

X, M = make_shapes(CFG["unet_images"])
n_val = max(16, len(X) // 5)
Xtr, Mtr = X[:len(X) - n_val].to(DEVICE), M[:len(M) - n_val].to(DEVICE)
Xva, Mva = X[len(X) - n_val:].to(DEVICE), M[len(M) - n_val:].to(DEVICE)

unet = TinyUNet().to(DEVICE)
opt = optim.Adam(unet.parameters(), lr=2e-3)
lossfn = nn.BCEWithLogitsLoss()
t0 = time.time()
for epoch in range(CFG["unet_epochs"]):
    unet.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 16):
        xb, mb = Xtr[perm[i:i + 16]], Mtr[perm[i:i + 16]]
        opt.zero_grad()
        loss = lossfn(unet(xb), mb)
        loss.backward()
        opt.step()
    unet.eval()
    with torch.no_grad():
        val = lossfn(unet(Xva), Mva).item()
    print(f"epoch {epoch+1}/{CFG['unet_epochs']} — val BCE loss {val:.4f}")
print(f"⏱️ Tiny U-Net trained in {time.time()-t0:.0f}s")

unet.eval()
with torch.no_grad():
    preds = (unet(Xva[:4]).sigmoid() > 0.5).float().cpu()
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for i in range(4):
    axes[0, i].imshow(Xva[i].cpu().permute(1, 2, 0)); axes[0, i].set_title("input", fontsize=9)
    axes[1, i].imshow(Mva[i, 0].cpu(), cmap="gray"); axes[1, i].set_title("ground truth", fontsize=9)
    axes[2, i].imshow(preds[i, 0], cmap="gray"); axes[2, i].set_title("prediction", fontsize=9)
    for ax in (axes[0, i], axes[1, i], axes[2, i]):
        ax.axis("off")
fig.suptitle("Tiny U-Net predictions")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "unet_predictions.png", bbox_inches="tight")
plt.show()

## 🏁 Wrap-up

**What you did today — the real skills:**
1. Ran a deep learning notebook on a **free cloud GPU** (no local installs).
2. **Downloaded** a benchmark dataset (CIFAR-10) and prepared it with `DataLoader`.
3. **Trained** a CNN from scratch — and watched accuracy climb.
4. **Downloaded pretrained weights** and **fine-tuned** a ResNet18 (transfer learning).
5. **Ran inference** with pretrained **detection** and **segmentation** models.
6. Put **your own image** through all three tasks and saved the outputs.

**Take it home:**
- Your results are in `outputs/`. Run the next cell to zip them.
- Run `local_cpu/participant_worksheet.py` next to those files for the offline analysis part of the lab.

**Next session (Session 2):** generative AI — diffusion models, text-to-image, image editing, multimodal vision, and the ethics of all of it.

**Credits & licenses:** sample images come from [Ultralytics](https://ultralytics.com/images/) (`bus`, `zidane`), [PyTorch Hub](https://github.com/pytorch/hub) (`dog`), [scikit-image](https://scikit-image.org/) (`astronaut`, `chelsea`), and [COCO val2017](http://cocodataset.org) (CC-BY 4.0). Models: torchvision (BSD-3). CIFAR-10: Krizhevsky, Nair & Hinton.

In [ ]:
# ---- Package your outputs ------------------------------------------------
zip_path = shutil.make_archive("session1_outputs", "zip", OUTPUT_DIR)
print("📦 Saved:", zip_path)

if IN_COLAB:
    from google.colab import files
    print("⬇️  A download should start. If not, run: files.download('session1_outputs.zip')")
    files.download("session1_outputs.zip")

print("\n✅ Session 1 lab complete — see you in the Fun Zone results!")